# Customer Churn Prediction — Simple Machine Learning Project

**Goal:** Predict whether a telecom customer will leave (Churn = Yes) using four classification algorithms: KNN, Naive Bayes, SVM, and Decision Tree.

> This notebook uses straightforward code and short explanations. Run cells from top to bottom.


## 1. Load and Inspect the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(df.shape)
df.info()


In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
# Convert TotalCharges to numbers; blank entries become NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing values with the median
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Check the Churn distribution
df['Churn'].value_counts()


## 2. Exploratory Data Analysis (EDA)

### 2.1 Univariate Analysis

In [ ]:
sns.countplot(x='Churn', data=df)
plt.title('Customer Churn')
plt.show()


In [ ]:
# Histograms for numerical columns
df[['tenure', 'MonthlyCharges', 'TotalCharges']].hist(figsize=(10, 4))
plt.tight_layout()
plt.show()


In [ ]:
# Countplots for categorical columns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(x='Contract', data=df, ax=axes[0])
sns.countplot(x='InternetService', data=df, ax=axes[1])
sns.countplot(x='PaymentMethod', data=df, ax=axes[2])
axes[2].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()


### 2.2 Bivariate Analysis

In [ ]:
# Churn across contract types
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title('Churn by Contract')
plt.show()


In [ ]:
# Churn rate for each contract type (percentage)
df.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)


In [ ]:
# Churn by tenure groups
tenure_bins = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 72],
                     labels=['0-12', '13-24', '25-48', '49-72'])
tenure_churn = df.groupby(tenure_bins, observed=False)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()

sns.barplot(x='tenure', y='Churn', data=tenure_churn)
plt.ylabel('Churn Rate (%)')
plt.title('Churn Rate by Tenure')
plt.show()


In [ ]:
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges vs Churn')
plt.show()


In [ ]:
# Correlation of numerical features
corr = df[['tenure', 'MonthlyCharges', 'TotalCharges']].corr()
sns.heatmap(corr, annot=True)
plt.show()


## 3. Prepare the Data

### 3.1 Remove the ID Column and Create Simple Features

In [ ]:
# customerID is just an identifier
df = df.drop(columns=['customerID'])

# Group customers by tenure
df['tenure_group'] = pd.cut(df['tenure'], bins=[-1, 12, 36, 60, np.inf],
                            labels=['New', 'Mid', 'Senior', 'Loyal'])

# Count how many add-on services each customer has
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']
df['num_services'] = (df[services] == 'Yes').sum(axis=1)

# 1 if they use automatic payment, otherwise 0
df['AutoPay'] = df['PaymentMethod'].apply(
    lambda x: 1 if 'automatic' in str(x).lower() else 0
)


### 3.2 Convert Categories into Numbers

In [ ]:
# Convert Yes/No columns to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Convert gender and MultipleLines
df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No').map({'Yes': 1, 'No': 0})

# Convert tenure group in its natural order
df['tenure_group'] = df['tenure_group'].map(
    {'New': 0, 'Mid': 1, 'Senior': 2, 'Loyal': 3}
)

# Remove service columns used to create num_services
df = df.drop(columns=services)

# Convert remaining categorical columns into dummy columns
df = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'],
                    drop_first=True, dtype=int)

print(df.shape)


### 3.3 Split into Training and Testing Data

We keep 20% of the data for testing. `stratify=y` helps keep the churn/non-churn proportions similar in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(X_train.shape)
print(X_test.shape)


### 3.4 Scale Numerical Features

Scaling is especially important for KNN and SVM.

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_services']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])


### 3.5 Handle Class Imbalance with SMOTE

SMOTE is applied only to the training data, not to the test data.

In [ ]:
from imblearn.over_sampling import SMOTE

print('Before SMOTE:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print('After SMOTE:')
print(pd.Series(y_train_smote).value_counts())


## 4. K-Nearest Neighbours (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_smote, y_train_smote)

y_pred_knn = knn.predict(X_test_scaled)
y_prob_knn = knn.predict_proba(X_test_scaled)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_pred_knn), 3))
print('Precision:', round(precision_score(y_test, y_pred_knn), 3))
print('Recall:', round(recall_score(y_test, y_pred_knn), 3))
print('F1:', round(f1_score(y_test, y_pred_knn), 3))
print('AUC:', round(roc_auc_score(y_test, y_prob_knn), 3))


In [ ]:
# Try a few values of k and compare their F1 scores
k_values = [3, 5, 7, 9]
f1_scores = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_smote, y_train_smote)
    pred = model.predict(X_test_scaled)
    f1_scores.append(f1_score(y_test, pred))

plt.plot(k_values, f1_scores, marker='o')
plt.xlabel('k')
plt.ylabel('F1 Score')
plt.title('KNN: k vs F1')
plt.show()


In [ ]:
# Confusion matrix
sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot=True, fmt='d')
plt.title('KNN Confusion Matrix')
plt.show()

# ROC curve
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)
plt.plot(fpr_knn, tpr_knn, label='KNN')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()


## 5. Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train_smote, y_train_smote)

y_pred_nb = nb.predict(X_test_scaled)
y_prob_nb = nb.predict_proba(X_test_scaled)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_pred_nb), 3))
print('Precision:', round(precision_score(y_test, y_pred_nb), 3))
print('Recall:', round(recall_score(y_test, y_pred_nb), 3))
print('F1:', round(f1_score(y_test, y_pred_nb), 3))
print('AUC:', round(roc_auc_score(y_test, y_prob_nb), 3))


In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred_nb), annot=True, fmt='d')
plt.title('Naive Bayes Confusion Matrix')
plt.show()

fpr_nb, tpr_nb, _ = roc_curve(y_test, y_prob_nb)
plt.plot(fpr_nb, tpr_nb, label='Naive Bayes')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()


## 6. Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', C=1, probability=True, random_state=42)
svm.fit(X_train_smote, y_train_smote)

y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_pred_svm), 3))
print('Precision:', round(precision_score(y_test, y_pred_svm), 3))
print('Recall:', round(recall_score(y_test, y_pred_svm), 3))
print('F1:', round(f1_score(y_test, y_pred_svm), 3))
print('AUC:', round(roc_auc_score(y_test, y_prob_svm), 3))


In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d')
plt.title('SVM Confusion Matrix')
plt.show()

fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)
plt.plot(fpr_svm, tpr_svm, label='SVM')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()


## 7. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt.fit(X_train_scaled, y_train)

y_pred_dt = dt.predict(X_test_scaled)
y_prob_dt = dt.predict_proba(X_test_scaled)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_pred_dt), 3))
print('Precision:', round(precision_score(y_test, y_pred_dt), 3))
print('Recall:', round(recall_score(y_test, y_pred_dt), 3))
print('F1:', round(f1_score(y_test, y_pred_dt), 3))
print('AUC:', round(roc_auc_score(y_test, y_prob_dt), 3))


In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d')
plt.title('Decision Tree Confusion Matrix')
plt.show()


In [ ]:
# Show only the first 3 levels so the tree is easy to read
plt.figure(figsize=(16, 8))
plot_tree(dt, max_depth=3, feature_names=X.columns,
          class_names=['No Churn', 'Churn'], filled=True)
plt.show()


In [ ]:
# Which features were most useful for the tree?
importance = pd.Series(dt.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

importance.head(10).plot(kind='bar')
plt.title('Top 10 Feature Importances')
plt.show()


## 8. Compare the Four Models

In [ ]:
# Create a simple table to compare the models
comparison = pd.DataFrame({
    'Model': ['KNN', 'Naive Bayes', 'SVM', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_dt)
    ],
    'Precision': [
        precision_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_svm),
        precision_score(y_test, y_pred_dt)
    ],
    'Recall': [
        recall_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_nb),
        recall_score(y_test, y_pred_svm),
        recall_score(y_test, y_pred_dt)
    ],
    'F1': [
        f1_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_svm),
        f1_score(y_test, y_pred_dt)
    ],
    'AUC': [
        roc_auc_score(y_test, y_prob_knn),
        roc_auc_score(y_test, y_prob_nb),
        roc_auc_score(y_test, y_prob_svm),
        roc_auc_score(y_test, y_prob_dt)
    ]
})

comparison.sort_values('Recall', ascending=False)


In [ ]:
# Plot all model ROC curves together
plt.plot(fpr_knn, tpr_knn, label='KNN')
plt.plot(fpr_nb, tpr_nb, label='Naive Bayes')
plt.plot(fpr_svm, tpr_svm, label='SVM')
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
plt.plot(fpr_dt, tpr_dt, label='Decision Tree')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.show()


In [ ]:
# Compare precision and recall
comparison.set_index('Model')[['Precision', 'Recall']].plot(kind='bar')
plt.ylabel('Score')
plt.title('Precision vs Recall')
plt.show()


## 9. SMOTE vs Balanced Class Weight

We compare the Decision Tree trained with SMOTE and a Decision Tree using `class_weight='balanced'`.

In [ ]:
# Decision Tree trained on SMOTE data
dt_smote = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_smote.fit(X_train_smote, y_train_smote)
pred_smote = dt_smote.predict(X_test_scaled)

# Decision Tree using class weights on original training data
dt_balanced = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt_balanced.fit(X_train_scaled, y_train)
pred_balanced = dt_balanced.predict(X_test_scaled)

print('Recall with SMOTE:', round(recall_score(y_test, pred_smote), 3))
print('Recall with class_weight:', round(recall_score(y_test, pred_balanced), 3))


## 10. Simple Error Analysis

False negatives are churners that the model missed.

In [ ]:
# Find customers who actually churned but were predicted not to churn
results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_dt})
false_negatives = results[(results['Actual'] == 1) & (results['Predicted'] == 0)]

print('Number of missed churners:', len(false_negatives))


## 11. Save the Model

This saves the trained Decision Tree model for later use.

In [ ]:
import joblib

joblib.dump(dt, 'churn_model.pkl')
print('Model saved!')


## Short Business Conclusion

After comparing the four models, use the comparison table to decide which model is most useful. In churn prediction, **Recall** is important because a False Negative means a customer who is about to leave was missed. Consider Recall alongside Precision: a retention team has limited time and cannot contact every customer flagged by the model.